# 03 · Đánh giá và xuất mô hình

Gắn output 01_DATA v4 và output 02_TRAIN Cau Truc đã chạy. Có thể gắn bộ test ngoài có manifest hợp lệ và tệp logits xuất từ Unity. Báo cáo hiển thị phần chưa có dữ liệu đo.


### Đưa mô hình vào app học tiếng Anh

Unity đọc `animal_mobilenetv3_fp32.onnx` và áp dụng tiền xử lý, thứ tự lớp, temperature và ngưỡng theo `preprocess.json`. Khi ảnh được nhận diện chắc chắn là một trong 9 con vật, app có thể hiện tên tiếng Anh và spawn Prefab sau khi người dùng xác nhận. Với `background` hoặc dưới ngưỡng, chuyển sang hướng dẫn chụp lại hoặc menu chọn thủ công.

Bản này có 50 ảnh và logits Python để đối chiếu. Kết quả đã lưu cho thấy Tiger F1 **92,06%** so với mục tiêu **93%**, Goat đạt **86,54%** khi bật TTA. Test ngoài và các phép đo trên Unity vẫn cần dữ liệu thực tế.


# Notebook 3 Hiệu chỉnh đánh giá và xuất

Theo phần đánh giá và triển khai trong file. Không bịa số liệu cho tập ngoài, Unity hoặc điện thoại khi chưa có dữ liệu đo.

## Ghi chú báo cáo

**Vai trò.** Notebook đánh giá `targeted_stage2.pth` từ 02_TRAIN v3 trên dữ liệu 01_DATA v4, hiệu chỉnh độ tin cậy và xuất ONNX.

**Kết quả test Open Images.** Không TTA: Top-1 93,80%, macro-F1 93,78%, Top-3 99,17%. TTA: Top-1 94,01%, macro-F1 93,97%, Top-3 99,17%. Recall background 92,00% ở cả hai chế độ. Goat F1 84,91% không TTA và 86,54% có TTA. Tiger F1 92,06% ở cả hai chế độ, dưới mốc 93%.

**Phạm vi.** Test Open Images đã được dùng để phân tích lỗi. Tập ngoài độc lập, kết quả logits Unity, độ trễ và thời gian spawn trên điện thoại còn cần đo.

[Nguồn Kaggle, phiên bản 2](https://www.kaggle.com/code/ngaqyh/03-evaluate?scriptVersionId=368347531)

## 01 Phụ thuộc và nối output dữ liệu huấn luyện

Sửa TRAIN và DATA tới đúng hai output đã Add Input. Nạp checkpoint đã chọn và đúng thứ tự 10 lớp. Tiền xử lý giống Notebook 2.

**Giải thích cell.** Kiểm tra manifest khớp dữ liệu huấn luyện và checkpoint khớp thứ tự 10 lớp. EXIF, resize và normalize giống Notebook 2.

In [1]:
# Khôi phục kiến trúc và trọng số đã chọn từ Notebook 02. Khi bật TTA, chạy ảnh gốc và ảnh lật ngang rồi lấy trung bình logits.
import sys, subprocess
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'onnx',
    'onnxruntime',
    'onnxconverter-common',
    'scikit-learn',
    'pandas',
    'matplotlib',
    'pillow',
    'imagehash'
])
from pathlib import Path
import json, shutil, hashlib
import numpy as np, pandas as pd
import torch, torch.nn as nn
from PIL import Image, ImageOps
from torchvision import transforms as T
from torchvision.transforms import InterpolationMode
from torchvision.models import mobilenet_v3_large
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
TRAIN = Path('/kaggle/input/notebooks/ngaqyh/02-train-cau-truc/animal_training')
DATA = Path('/kaggle/input/notebooks/ngaqyh/01-data/animal_data')
cfg = json.loads((TRAIN / 'training_config.json').read_text())
manifest_paths = [DATA / 'dataset_manifest.csv']
assert manifest_paths[0].is_file(), 'Chưa có dữ liệu Notebook 1'
OUT = Path('/kaggle/working/animal_evaluation')
OUT.mkdir(parents=True, exist_ok=True)
for d in ['metrics', 'charts', 'export', 'unity50']:
    (OUT / d).mkdir(exist_ok=True)
CLASSES = cfg['classes']
C = len(CLASSES)
INDEX = {c: i for i, c in enumerate(CLASSES)}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tf = T.Compose([
    T.Resize((224, 224), interpolation=InterpolationMode.BILINEAR, antialias=True),
    T.ToTensor(),
    T.Normalize(cfg['mean'], cfg['std'])
])
df = pd.read_csv(manifest_paths[0], dtype={'image_id': str})
df['path'] = df.relpath.map(lambda p: str(DATA / p))

def build(path):
    m = mobilenet_v3_large(weights=None)
    m.classifier[-1] = nn.Sequential(nn.Dropout(0.2), nn.Linear(m.classifier[-1].in_features, C))
    ck = torch.load(path, map_location='cpu', weights_only=False)
    assert ck['classes'] == CLASSES
    m.load_state_dict(ck['state_dict'])
    return m.eval().to(DEVICE)

class Images(Dataset):

    def __init__(self, sub):
        self.sub = sub.reset_index(drop=True)

    def __len__(self):
        return len(self.sub)

    def __getitem__(self, i):
        r = self.sub.iloc[i]
        with Image.open(r.path) as im:
            x = tf(ImageOps.exif_transpose(im).convert('RGB'))
        return (x, INDEX[r.label])

@torch.inference_mode()
def infer(model, sub, tta=False):
    zs = []
    ys = []
    for x, y in DataLoader(Images(sub), batch_size=32, num_workers=2):
        x = x.to(DEVICE)
        z = model(x)
        if tta:
            z = (z + model(torch.flip(x, [-1]))) / 2
        zs.append(z.cpu())
        ys.append(y)
    return (torch.cat(zs).float(), torch.cat(ys))
CHECKPOINT = TRAIN / 'checkpoints' / 'targeted_stage2.pth'
assert CHECKPOINT.is_file(), 'Thiếu checkpoint fine-tune 02_TRAIN v3'
assert manifest_paths[0].read_bytes() == (TRAIN / 'data_manifest.csv').read_bytes(), 'Manifest khác dữ liệu đã train'
assert df.image_id.is_unique and df.path.map(lambda p: Path(p).is_file()).all()
print('Checkpoint:', CHECKPOINT.name)
print('Số ảnh:', df.split.value_counts().to_dict())
model = build(CHECKPOINT)


Checkpoint: targeted_stage2.pth
Số ảnh: {'train': 2000, 'test': 484, 'val': 390, 'test_cartoon': 130}


## 02 Temperature scaling riêng trên val và ngưỡng không chắc riêng từng TTA

Tìm temperature bằng val để giảm NLL. Chọn ngưỡng từ precision/coverage trên val; chỉ công nhận khi accuracy phần nhận diện≥95% và coverage≥80%. Hiệu chỉnh riêng single và TTA. Nếu không đạt, báo không có ngưỡng phù hợp.

**Giải thích cell.** Tìm nhiệt độ T bằng NLL trên val. Chọn ngưỡng độ tin cậy dựa trên đường accuracy/coverage của val. Mỗi chế độ suy luận có một cấu hình.

In [2]:
# Dùng validation để tìm temperature và ngưỡng không chắc. Temperature đổi độ tin cậy; với T dương, thứ tự logits giữ nguyên.
def softmax(z):
    return torch.softmax(z, dim=1).numpy()

def ece(probs, y, bins=15):
    confidence = probs.max(1)
    correct = probs.argmax(1) == np.asarray(y)
    value = 0.0
    for lo, hi in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        mask = (confidence > lo) & (confidence <= hi)
        if mask.any():
            value += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(value)

def calibration(z, y, key):
    z, y = z.clone(), y.clone()
    logT = torch.zeros(1, requires_grad=True)
    optimizer = torch.optim.LBFGS([logT], lr=0.1, max_iter=100, line_search_fn='strong_wolfe')

    def closure():
        optimizer.zero_grad()
        loss = nn.functional.cross_entropy(z / logT.exp().clamp(0.05, 20), y)
        loss.backward()
        return loss
    optimizer.step(closure)
    temp = float(logT.exp().clamp(0.05, 20).detach())
    p = softmax(z / temp)
    conf = p.max(1)
    correct = p.argmax(1) == y.numpy()
    rows = []
    for threshold in np.unique(np.r_[0, conf, 1]):
        keep = conf >= threshold
        rows.append(dict(
            threshold=float(threshold),
            coverage=float(keep.mean()),
            precision=float(correct[keep].mean()) if keep.any() else None,
            accepted=int(keep.sum())
        ))
    table = pd.DataFrame(rows)
    table.to_csv(OUT / 'metrics' / f'precision_coverage_{key}.csv', index=False)
    feasible = table[(table.precision >= 0.95) & (table.accepted > 0)]
    point = None if feasible.empty else feasible.sort_values(['coverage', 'threshold'], ascending=[False, True]).iloc[0]
    chosen = None if point is None else float(point.threshold)
    table.plot(x='coverage', y='precision', legend=False)
    plt.axhline(0.95, color='r', ls='--')
    plt.axvline(0.8, color='gray', ls='--')
    plt.ylim(0, 1.01)
    plt.tight_layout()
    plt.savefig(OUT / 'charts' / f'precision_coverage_{key}.png', dpi=150)
    plt.close()
    return dict(
        temperature=temp,
        threshold=chosen,
        target_met=point is not None and float(point.coverage) >= 0.8,
        coverage=None if point is None else float(point.coverage),
        accepted_accuracy=None if point is None else float(point.precision),
        ece_before=ece(softmax(z), y),
        ece_after=ece(p, y),
        nll_before=float(nn.functional.cross_entropy(z, y)),
        nll_after=float(nn.functional.cross_entropy(z / temp, y))
    )
cal = {}
for tta in [False, True]:
    key = 'tta' if tta else 'single'
    z, y = infer(model, df[df.split == 'val'], tta)
    cal[key] = calibration(z, y, key)
(OUT / 'export' / 'calibration.json').write_text(json.dumps(cal, indent=2))


690

## 03 Test trong phân phối cartoon và test ngoài có nguồn độc lập

Đánh giá test Open Images, cartoon và test ngoài. Test ngoài do bạn gom từ nguồn khác, CSV có relpath,label,image_id,source,license. Nếu chưa có thì báo chưa đánh giá. Xuất accuracy, F1 từng lớp, confusion matrix và ảnh lỗi; so có/không TTA.

**Giải thích cell.** Đo trên 484 ảnh Open Images và 130 ảnh cartoon, với và không TTA. TTA lấy trung bình logits ảnh gốc và ảnh lật ngang. Xuất confusion matrix, F1 theo lớp và danh sách lỗi.

In [3]:
# Đánh giá từng tập riêng: test trong phân phối, test ngoài và cartoon. Đọc macro-F1 cùng confusion matrix để biết lớp nào hay bị nhầm.
EXTERNAL_MANIFEST = Path('/kaggle/input/external-test/external_manifest.csv')
external_paths = [EXTERNAL_MANIFEST] if EXTERNAL_MANIFEST.is_file() else sorted(
    Path('/kaggle/input').glob('**/external_manifest.csv')
)
if len(external_paths) > 1:
    raise ValueError('Có nhiều external_manifest.csv. Hãy chọn đường dẫn ở EXTERNAL_MANIFEST.')
external_status = 'missing'
if external_paths:
    ep = external_paths[0]
    ext = pd.read_csv(ep, dtype={'image_id': str})
    assert {'relpath', 'label', 'image_id', 'source', 'license'} <= set(ext)
    assert ext.label.isin(CLASSES).all() and ext.license.notna().all()
    assert not ext.source.str.contains('OpenImages|Open Images', case=False, regex=True).any()
    assert set(ext.image_id).isdisjoint(set(df.image_id))
    assert ext.groupby('label').size().reindex(CLASSES, fill_value=0).min() >= 30
    ext['path'] = ext.relpath.map(lambda p: str(ep.parent / p))
    assert ext.path.map(lambda p: Path(p).is_file()).all()
    import imagehash
    known = []
    for p in df.path:
        with Image.open(p) as im:
            known.append(int(str(imagehash.phash(ImageOps.exif_transpose(im).convert('RGB'))), 16))
    for p in ext.path:
        with Image.open(p) as im:
            h = int(str(imagehash.phash(ImageOps.exif_transpose(im).convert('RGB'))), 16)
        assert all(((h ^ k).bit_count() > 6 for k in known)), 'Test ngoài trùng gần dữ liệu Open Images'
    ext['split'] = 'test_external'
    df = pd.concat([df, ext], ignore_index=True)
    external_status = 'available'

def metrics(z, y, temp=1.0, threshold=None):
    p = softmax(z / temp)
    pred = p.argmax(1)
    yn = y.numpy()
    top3 = np.argsort(p, axis=1)[:, -3:]
    bg = yn == INDEX['background']
    accept = np.zeros(len(y), dtype=bool) if threshold is None else p.max(1) >= threshold
    return dict(
        accuracy=float(accuracy_score(yn, pred)),
        macro_f1=float(f1_score(yn, pred, labels=sorted(set(yn)), average='macro', zero_division=0)),
        top3=float(np.any(top3 == yn[:, None], axis=1).mean()),
        background_recall=float((pred[bg] == INDEX['background']).mean()) if bg.any() else None,
        ece=ece(p, y),
        coverage=float(accept.mean()) if threshold is not None else None,
        accepted_accuracy=float((pred[accept] == yn[accept]).mean()) if accept.any() else None
    )
summary = []
for split in ['test', 'test_cartoon', 'test_external']:
    sub = df[df.split == split].reset_index(drop=True)
    if sub.empty:
        print('CHƯA CÓ', split)
        continue
    for tta in [False, True]:
        key = 'tta' if tta else 'single'
        z, y = infer(model, sub, tta)
        cc = cal[key]
        met = metrics(z, y, cc['temperature'], cc['threshold'])
        summary.append(dict(split=split, mode=key, **met))
        np.savez(
            OUT / 'metrics' / f'{split}_{key}.npz',
            logits=z.numpy(),
            y=y.numpy(),
            image_ids=sub.image_id.to_numpy(dtype=str)
        )
        preds = z.argmax(1).numpy()
        report = classification_report(
            y,
            preds,
            labels=list(range(C)),
            target_names=CLASSES,
            output_dict=True,
            zero_division=0
        )
        (OUT / 'metrics' / f'{split}_{key}_per_class.json').write_text(json.dumps(report, indent=2))
        cm = confusion_matrix(y, preds, labels=list(range(C)))
        pd.DataFrame(
            cm,
            index=CLASSES,
            columns=CLASSES
        ).to_csv(OUT / 'metrics' / f'{split}_{key}_confusion.csv')
        fig, ax = plt.subplots(figsize=(8, 7))
        img = ax.imshow(cm, cmap='Blues')
        ax.set_xticks(range(C), CLASSES, rotation=60, ha='right')
        ax.set_yticks(range(C), CLASSES)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        fig.colorbar(img, ax=ax)
        fig.tight_layout()
        fig.savefig(OUT / 'charts' / f'{split}_{key}_confusion.png', dpi=150)
        plt.close(fig)
        errors = sub.copy()
        errors['predicted'] = [CLASSES[i] for i in preds]
        errors['confidence'] = softmax(z / cc['temperature']).max(1)
        errors[errors.predicted != errors.label].to_csv(OUT / 'metrics' / f'{split}_{key}_errors.csv', index=False)
pd.DataFrame(summary).to_csv(OUT / 'metrics' / 'evaluation_summary.csv', index=False)


CHƯA CÓ test_external


## 04 So sánh stage1 stage2 CLIP ba seed learning curve và ablation trên test

So stage1/stage2, ba seed, learning curve, ablation và CLIP trên test. Báo mean±std theo seed. Không dùng các kết quả test này để chọn seed hay ngưỡng.

**Giải thích cell.** So sánh checkpoint và CLIP cùng test ID. Đây là bảng thực nghiệm. Checkpoint chính được chọn trước bằng val.

In [4]:
# Đặt kết quả CLIP cạnh các phiên bản MobileNet để đọc chênh lệch trên cùng tập và chế độ đo.
comparisons = []
for p in sorted((TRAIN / 'checkpoints').glob('*.pth')):
    if p.name.startswith('selected_'):
        continue
    m = build(p)
    z, y = infer(m, df[df.split == 'test'])
    met = metrics(z, y)
    comparisons.append(dict(checkpoint=p.name, **met))
    del m
comparison = pd.DataFrame(comparisons)
comparison.to_csv(OUT / 'metrics' / 'experiment_test_comparison.csv', index=False)
for stage in [1, 2]:
    seeds = comparison[comparison.checkpoint.str.match('seed_\\d+_stage' + str(stage) + '\\.pth')]
    seeds[[
        'accuracy',
        'macro_f1',
        'top3',
        'background_recall'
    ]].agg(['mean', 'std']).to_csv(OUT / 'metrics' / f'stage{stage}_three_seed_test_summary.csv')
clip = np.load(TRAIN / 'metrics' / 'clip_test.npz')
assert np.array_equal(clip['image_ids'], df.loc[df.split == 'test', 'image_id'].to_numpy(dtype=str)), 'CLIP dùng thứ tự ảnh test khác'
clipz = torch.tensor(clip['logits'])
clipy = torch.tensor(clip['y'])
clipmet = metrics(clipz, clipy)
(OUT / 'metrics' / 'clip_test_summary.json').write_text(json.dumps(clipmet, indent=2))
print('CLIP test', clipmet)


CLIP test {'accuracy': 0.9008264462809917, 'macro_f1': 0.8947158944796388, 'top3': 0.9855371900826446, 'background_recall': 0.54, 'ece': 0.03128636517554273, 'coverage': None, 'accepted_accuracy': None}


## 05 Xuất ONNX FP32 đối chiếu logits và thử FP16 trên toàn test

Xuất ONNX input1×3×224×224, chạy onnxruntime và so logits với PyTorch, yêu cầu sai khác≤1e-4. Thử FP16 trên toàn test để so accuracy trước/sau; lỗi hỗ trợ FP16 phải ghi là chưa xác minh.

**Giải thích cell.** Xuất ONNX FP32/FP16 và kiểm tra ONNX Runtime. FP32 lệch logits tối đa khoảng 1,04e-5. FP16 khớp Top-1 với FP32 trên cả 484 ảnh test.

In [5]:
# Xuất ONNX FP32, so logits với PyTorch, rồi chuyển sang FP16 và đo mức khớp top-1 trên test.
import onnxruntime as ort
import onnx
from onnxconverter_common import float16
model = build(CHECKPOINT).cpu().eval()
torch.set_num_threads(2)
options = ort.SessionOptions()
options.intra_op_num_threads = 2
options.inter_op_num_threads = 1
# Tensor mẫu định nghĩa đầu vào ONNX: một ảnh RGB theo trật tự NCHW.
dummy = torch.zeros(1, 3, 224, 224)
fp32 = OUT / 'export' / 'animal_mobilenetv3_fp32.onnx'
torch.onnx.export(
    model,
    dummy,
    str(fp32),
    input_names=['images'],
    output_names=['logits'],
    opset_version=17,
    dynamo=False
)
onnx.checker.check_model(onnx.load(str(fp32)))
session = ort.InferenceSession(str(fp32), sess_options=options, providers=['CPUExecutionProvider'])
test = df[df.split == 'test'].reset_index(drop=True)
ort_logits = []
pt_logits = []
tensor_rows = []
with torch.inference_mode():
    for i in range(len(test)):
        x, y = Images(test)[i]
        x = x.unsqueeze(0)
        pt = model(x).numpy()
        zz = session.run(None, {'images': x.numpy()})[0]
        pt_logits.append(pt[0])
        ort_logits.append(zz[0])
pt_logits = np.array(pt_logits)
ort_logits = np.array(ort_logits)
# So sai khác tuyệt đối lớn nhất giữa hai tập logits, trước softmax.
maxdiff = float(np.abs(pt_logits - ort_logits).max())
print(f'ONNX FP32 max abs diff: {maxdiff:.7g}')
fp16 = OUT / 'export' / 'animal_mobilenetv3_fp16.onnx'
fp16_report = {}
try:
    # FP16 giảm dung lượng mô hình; keep_io_types giữ giao diện đầu vào và đầu ra float32.
    onnx.save(float16.convert_float_to_float16(onnx.load(str(fp32)), keep_io_types=True), str(fp16))
    onnx.checker.check_model(onnx.load(str(fp16)))
    s16 = ort.InferenceSession(str(fp16), sess_options=options, providers=['CPUExecutionProvider'])
    z16 = []
    for i in range(len(test)):
        x, _ = Images(test)[i]
        z16.append(s16.run(None, {'images': x.unsqueeze(0).numpy()})[0][0])
    z16 = np.array(z16)
    y = test.label.map(INDEX).to_numpy()
    fp16_report = dict(
        status='tested',
        accuracy_fp32=float((ort_logits.argmax(1) == y).mean()),
        accuracy_fp16=float((z16.argmax(1) == y).mean()),
        max_abs_diff=float(np.abs(ort_logits - z16).max()),
        top1_agreement=float((ort_logits.argmax(1) == z16.argmax(1)).mean())
    )
except Exception as exc:
    fp16_report = dict(status='not_verified', error=str(exc))
export_report = dict(
    fp32_max_abs_diff=maxdiff,
    fp32_logits_passed=maxdiff <= 0.0001,
    fp32_mb=fp32.stat().st_size / 1000000.0,
    fp16_mb=fp16.stat().st_size / 1000000.0 if fp16.exists() else None,
    fp16=fp16_report
)
(OUT / 'export' / 'onnx_validation.json').write_text(json.dumps(export_report, indent=2))


/tmp/ipykernel_22/3542766935.py:11: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX FP32 max abs diff: 1.037121e-05


/usr/local/lib/python3.13/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 5.547115284471049e-12 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -1.517812438234234e-11 will be truncated to -1e-07
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 2.115027597504593e-10 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -1.697370369257989e-11 will be truncated to -1e-07
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 4.706539520095421e-13 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -1.544343320759866

315

## 06 Bộ 50 ảnh logits Python và hợp đồng tiền xử lý Unity

Xuất 50 ảnh, thứ tự lớp, tensor đã tiền xử lý và logits Python để so Unity. RGB, EXIF, resize, mean/std, NCHW và TTA đều ghi rõ. Khi Unity xuất logits cùng thứ tự, kiểm tra sai khác≤1e-3 và top1 khớp100%. Tốc độ điện thoại phải đo trên máy thật.

**Giải thích cell.** Lưu 50 ảnh (5 mỗi lớp), tensor đầu vào và logits Python cho Unity. Phần Unity thật cần chạy bằng cùng cấu hình tiền xử lý.

In [6]:
# Xuất 5 ảnh mỗi lớp và logits Python làm mốc cho Unity. Ghép kết quả bằng image_id để thứ tự dòng CSV không làm lệch phép so.
probe = pd.concat([g.sample(n=min(5, len(g)), random_state=42) for _, g in test.groupby('label')]).reset_index(drop=True)
assert len(probe) == 50, 'Cần 5 ảnh mỗi lớp cho bộ Unity 50'
xs = []
zs = []
zt = []
records = []
with torch.inference_mode():
    for i, r in probe.iterrows():
        dst = OUT / 'unity50' / (r.image_id + '.jpg')
        shutil.copy2(r.path, dst)
        x, _ = Images(probe)[i]
        x = x.unsqueeze(0)
        z = model(x)
        ztta = (z + model(x.flip([-1]))) / 2
        xs.append(x.numpy()[0])
        zs.append(z.numpy()[0])
        zt.append(ztta.numpy()[0])
        records.append(dict(
            filename=dst.name,
            image_id=r.image_id,
            label=r.label,
            top1=CLASSES[int(z.argmax())]
        ))
np.save(OUT / 'unity50' / 'preprocessed_nchw.npy', np.array(xs))
np.save(OUT / 'unity50' / 'python_logits.npy', np.array(zs))
np.save(OUT / 'unity50' / 'python_tta_logits.npy', np.array(zt))
pd.DataFrame(records).to_csv(OUT / 'unity50' / 'images.csv', index=False)
pd.DataFrame(zs, columns=CLASSES).to_csv(OUT / 'unity50' / 'python_logits.csv', index=False)
# Ghi lại quy tắc tiền xử lý, thứ tự lớp và ngưỡng để Unity dùng cùng quy tắc với Python.
contract = dict(
    checkpoint=CHECKPOINT.name,
    input_shape=[1, 3, 224, 224],
    dtype='float32',
    layout='NCHW',
    color='RGB',
    range_before_normalize=[0, 1],
    mean=cfg['mean'],
    std=cfg['std'],
    resize='Pillow Resize((224,224)), bilinear, antialias=True; full image, no center crop',
    exif='ImageOps.exif_transpose before RGB',
    tta='mean raw logits of original and horizontal flip, then divide by T and softmax; use TTA-specific temperature/threshold',
    classes=CLASSES,
    calibration=cal,
    logits_tolerance_unity=0.001,
    top1_agreement_required=1.0,
    flow='Chọn ảnh → EXIF → RGB resize normalize → inference → top3 → menu nếu chưa có ngưỡng hoặc dưới ngưỡng → xác nhận → spawn Prefab'
)
(OUT / 'export' / 'preprocess.json').write_text(json.dumps(contract, ensure_ascii=False, indent=2))
UNITY_LOGITS = Path('/kaggle/input/unity-check/unity_logits.csv')
unity = [UNITY_LOGITS] if UNITY_LOGITS.is_file() else sorted(Path('/kaggle/input').glob('**/unity_logits.csv'))
if len(unity) > 1:
    raise ValueError('Có nhiều unity_logits.csv. Hãy chọn đường dẫn ở UNITY_LOGITS.')
unity_result = None
# Chỉ chấm phép so Python–Unity khi có CSV logits từ Unity cho đúng 50 ảnh.
if unity:
    unity_frame = pd.read_csv(unity[0], dtype={'image_id': str})
    assert 'image_id' in unity_frame and unity_frame.image_id.is_unique, 'Unity cần một dòng mỗi image_id'
    ids = probe.image_id.to_list()
    assert set(unity_frame.image_id) == set(ids), 'Unity cần đúng 50 ID ảnh từ images.csv'
    u = unity_frame.set_index('image_id').loc[ids, CLASSES].to_numpy(dtype=float)
    assert np.isfinite(u).all(), 'Logits Unity phải là số hữu hạn'
    p = np.array(zs)
    assert u.shape == p.shape
    diff = float(np.abs(u - p).max())
    agree = float((u.argmax(1) == p.argmax(1)).mean())
    unity_result = dict(max_abs_diff=diff, top1_agreement=agree, passed=diff <= 0.001 and agree == 1.0)
    (OUT / 'metrics' / 'unity_comparison.json').write_text(json.dumps(unity_result, indent=2))
    print('Sai khác logits Unity:', diff, '| Top-1 khớp:', f'{agree:.2%}')
else:
    print('Unity và độ trễ điện thoại: chưa đo; cần chạy trên thiết bị thật.')


Unity và độ trễ điện thoại: chưa đo; cần chạy trên thiết bị thật.


## 07 Báo cáo đạt chưa đạt không thay mục tiêu bằng kết quả giả

Đối chiếu các mục tiêu trong phần kết quả cần đạt: accuracy/F1, background, cartoon/ngoài, ECE, coverage, dung lượng. Mục chưa có bằng chứng là chưa đánh giá, không tính là đạt. Nếu sai nhiều một cặp, bổ sung có chủ đích cho train và giữ test cố định.

**Giải thích cell.** Đối chiếu mục tiêu TRAIN.docx. Các kiểm tra per-class trong criteria.json mặc định là không TTA. Đọc test_tta_per_class.json để báo riêng kết quả TTA.

In [7]:
# Đối chiếu chỉ số đã đo với các ngưỡng yêu cầu. Chỉ kết luận cho những mục có đủ dữ liệu kiểm tra.
by = {(r['split'], r['mode']): r for r in summary}
r = by.get(('test', 'single'), {})
per = json.loads((OUT / 'metrics' / 'test_single_per_class.json').read_text())
checks = {
    'test_accuracy_90': r.get('accuracy', 0) >= 0.9,
    'test_macro_f1_90': r.get('macro_f1', 0) >= 0.9,
    'test_top3_97': r.get('top3', 0) >= 0.97,
    'background_recall_90': (r.get('background_recall') or 0) >= 0.9,
    'ece_val_05': cal['single']['ece_after'] <= 0.05,
    'precision_coverage_val': cal['single']['target_met'],
    'external_accuracy_85': None if external_status == 'missing' else by['test_external', 'single']['accuracy'] >= 0.85,
    'cartoon_accuracy_70': None if ('test_cartoon', 'single') not in by else by['test_cartoon', 'single']['accuracy'] >= 0.7,
    'fp32_logits_1e_4': export_report['fp32_logits_passed'],
    'fp16_accuracy_verified': export_report['fp16']['status'] == 'tested',
    'fp32_size_25mb': export_report['fp32_mb'] <= 25,
    'fp16_size_12mb': None if export_report['fp16_mb'] is None else export_report['fp16_mb'] <= 12,
    'unity_logits': None if unity_result is None else unity_result['passed'],
    'phone_latency_200ms': None,
    'select_to_spawn_3s': None,
    'manual_correction_20percent': None
}
for c in ['Dog', 'Cat', 'Penguin', 'Tiger', 'Chicken', 'Horse']:
    checks[c + '_f1_93'] = per[c]['f1-score'] >= 0.93
for c in ['Goat', 'Sheep', 'Deer']:
    checks[c + '_f1_85'] = per[c]['f1-score'] >= 0.85
clip_per = json.loads((TRAIN / 'metrics' / 'clip_test.json').read_text())
checks['beat_clip_background'] = per['background']['f1-score'] > clip_per['background']['f1-score']
checks['beat_clip_one_hard_pair'] = any((per[a]['f1-score'] + per[b]['f1-score'] > clip_per[a]['f1-score'] + clip_per[b]['f1-score'] for a, b in [
    ('Goat', 'Sheep'),
    ('Deer', 'Goat'),
    ('Deer', 'Sheep')
]))
s1 = comparison[comparison.checkpoint.str.match('seed_\\d+_stage1')].macro_f1.mean()
s2 = comparison[comparison.checkpoint.str.match('seed_\\d+_stage2')].macro_f1.mean()
checks['stage2_improves_stage1'] = bool(s2 > s1)
paired = comparison.set_index('checkpoint').macro_f1
checks['targeted_beats_seed42_stage1_test'] = bool(paired[CHECKPOINT.name] > paired['seed_42_stage1.pth'])
focused = dict(background_recall=per['background']['recall'], tiger_f1=per['Tiger']['f1-score'], goat_f1=per['Goat']['f1-score'])
report = dict(checkpoint=CHECKPOINT.name, thresholds=checks, evaluation=summary,
              calibration=cal, focused_test_metrics=focused)
(OUT / 'metrics' / 'criteria.json').write_text(json.dumps(report, indent=2))
shutil.make_archive(str(OUT / 'unity_export'), 'zip', OUT / 'export')
print('Save Version để giữ tất cả output.')


Save Version để giữ tất cả output.


## Kết quả đánh giá

Bảng bên dưới lưu số liệu từ phiên bản Kaggle trong bộ báo cáo. Chạy cell này để đọc lại các tệp kết quả.

In [8]:
# Hiển thị bảng tổng hợp và file bàn giao. Việc chạy trong Unity và đo độ trễ cần thực hiện trên điện thoại.
from pathlib import Path
import csv
import json
from IPython.display import Markdown, display

def bang(tieu_de, cot, hang):
    def dong(gia_tri):
        return "| " + " | ".join(map(str, gia_tri)) + " |"
    display(Markdown("\n".join([
        f"### {tieu_de}", "", dong(cot), dong(["---"] * len(cot)),
        *(dong(h) for h in hang),
    ])))

def phan_tram(value):
    if value is None or value == "":
        return "—"
    return f"{100 * float(value):.2f}%".replace(".", ",")

def doc_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

def doc_json(path):
    return json.loads(path.read_text(encoding="utf-8-sig"))

def tim_output(kaggle, thu_muc, tep):
    for folder in [kaggle, Path.cwd() / thu_muc, Path.cwd().parent / thu_muc]:
        if (folder / tep).is_file():
            return folder
    raise FileNotFoundError(f"Chạy các cell trước để tạo {tep}, hoặc mở notebook trong bộ báo cáo đã giải nén.")

folder = tim_output(Path("/kaggle/working/animal_evaluation"), "danh_gia", "metrics/evaluation_summary.csv")
metrics = folder / "metrics"
criteria = doc_json(metrics / "criteria.json")
display(Markdown(f"**Checkpoint được đánh giá:** `{criteria['checkpoint']}`"))
bang("Kết quả trên tập kiểm tra", ["Tập", "Chế độ", "Top-1", "Macro-F1", "Top-3", "Recall background"],
     [(r["split"], "TTA" if r["mode"] == "tta" else "Một lượt", phan_tram(r["accuracy"]),
       phan_tram(r["macro_f1"]), phan_tram(r["top3"]), phan_tram(r["background_recall"]))
      for r in doc_csv(metrics / "evaluation_summary.csv")])
single = doc_json(metrics / "test_single_per_class.json")
tta = doc_json(metrics / "test_tta_per_class.json")
bang("F1 theo lớp · test Open Images", ["Lớp", "Một lượt", "TTA", "Mục tiêu"],
     [(label, phan_tram(single[label]["f1-score"]), phan_tram(tta[label]["f1-score"]),
       "≥ 85%" if label in ("Goat", "Sheep", "Deer") else ("—" if label == "background" else "≥ 93%"))
      for label, value in single.items() if isinstance(value, dict) and "f1-score" in value and "avg" not in label])
bang("Hiệu chỉnh trên validation", ["Chế độ", "ECE sau hiệu chỉnh", "Ngưỡng", "Coverage", "Accuracy phần chấp nhận"],
     [(mode, f"{r['ece_after']:.4f}", phan_tram(r["threshold"]),
       phan_tram(r["coverage"]), phan_tram(r["accepted_accuracy"]))
      for mode, r in criteria["calibration"].items()])
checks = criteria["thresholds"]
onnx = folder / "export" / "onnx_validation.json"
if onnx.is_file():
    report = doc_json(onnx)
    fp16 = report["fp16"]
    bang("Kiểm tra ONNX", ["Chỉ số", "Kết quả"],
         [("Dung lượng FP32", f"{report['fp32_mb']:.2f} MB"),
          ("Dung lượng FP16", f"{report['fp16_mb']:.2f} MB" if report["fp16_mb"] is not None else "—"),
          ("Sai khác logits tối đa FP32", f"{report['fp32_max_abs_diff']:.2e}"),
          ("Top-1 FP16", phan_tram(fp16.get("accuracy_fp16"))),
          ("Tỷ lệ FP16 và FP32 cùng dự đoán Top-1", phan_tram(fp16.get("top1_agreement")))])
display(Markdown("**Test ngoài độc lập:** " + ("Chưa có kết quả" if checks.get("external_accuracy_85") is None
                                            else ("Đạt" if checks["external_accuracy_85"] else "Dưới mục tiêu"))))


**Checkpoint được đánh giá:** `targeted_stage2.pth`

### Kết quả trên tập kiểm tra

| Tập | Chế độ | Top-1 | Macro-F1 | Top-3 | Recall background |
| --- | --- | --- | --- | --- | --- |
| test | Một lượt | 93,80% | 93,78% | 99,17% | 92,00% |
| test | TTA | 94,01% | 93,97% | 99,17% | 92,00% |
| test_cartoon | Một lượt | 73,08% | 75,00% | 91,54% | — |
| test_cartoon | TTA | 73,08% | 74,60% | 91,54% | — |

### F1 theo lớp · test Open Images

| Lớp | Một lượt | TTA | Mục tiêu |
| --- | --- | --- | --- |
| Dog | 96,15% | 95,15% | ≥ 93% |
| Cat | 96,97% | 96,97% | ≥ 93% |
| Horse | 98,99% | 98,99% | ≥ 93% |
| Sheep | 88,42% | 88,42% | ≥ 85% |
| Goat | 84,91% | 86,54% | ≥ 85% |
| Deer | 92,78% | 94,95% | ≥ 85% |
| Tiger | 92,06% | 92,06% | ≥ 93% |
| Penguin | 100,00% | 100,00% | ≥ 93% |
| Chicken | 99,01% | 99,01% | ≥ 93% |
| background | 88,46% | 87,62% | — |

### Hiệu chỉnh trên validation

| Chế độ | ECE sau hiệu chỉnh | Ngưỡng | Coverage | Accuracy phần chấp nhận |
| --- | --- | --- | --- | --- |
| single | 0.0309 | 51,52% | 98,72% | 95,06% |
| tta | 0.0325 | 59,43% | 96,15% | 95,20% |

### Kiểm tra ONNX

| Chỉ số | Kết quả |
| --- | --- |
| Dung lượng FP32 | 16.85 MB |
| Dung lượng FP16 | 8.46 MB |
| Sai khác logits tối đa FP32 | 1.04e-05 |
| Top-1 FP16 | 93,80% |
| Tỷ lệ FP16 và FP32 cùng dự đoán Top-1 | 100,00% |

**Test ngoài độc lập:** Chưa có kết quả